In [ ]:
import requests
import pandas as pd
import sqlite3

In [ ]:
server = "https://environment.data.gov.uk/wales/bathing-waters/doc/bathing-water?_pageSize=1000&_view=bathing-water&_properties=latestProfile.countyName.name%2Cdistrict.alias%2ClatestSampleAssessment.followingSuspension.endOfSuspension%2ClatestSampleAssessment.sampleDateTime.ordinalYear%2ClatestComplianceAssessment.sampleYear.ordinalYear%2ClatestComplianceAssessment.assessmentQualifier%2ClatestComplianceAssessment.assessmentRegime&_lang=en%2Ccy%2Cnone"

In [ ]:
data = requests.get(server, headers={"Accept": "application/json, text/javascript, */*; q=0.01"})
result = data.json()['result']['items']

In [ ]:
df = pd.json_normalize(data.json()['result']['items'])
df["name"], df["alternate_name"] = [item[1]['_value']
                                    for item in df['name']], [item[0]['_value'] for item in df['name']]
df = df.rename(columns={
    "eubwidNotation": "id",
    "samplingPoint.lat": "lat",
    "samplingPoint.long": "lon"
})
df.set_index("id")

In [ ]:
locations = df[[
    "id", "name", "alternate_name", "lat", "lon"
]].copy().set_index("id")

In [ ]:
locations

In [ ]:
db = sqlite3.connect("../dataset.sqlite3")
locations.to_sql("locations", db, if_exists="append")
